In [1]:
import pandas as pd
from transformers import T5Tokenizer, Trainer , TrainingArguments , T5ForConditionalGeneration

In [2]:
train_data=pd.read_csv("data/samsum-train.csv")
val_data = pd.read_csv("data/samsum-validation.csv")

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_data.shape

(14732, 3)

In [5]:
val_data.shape

(818, 3)

In [6]:
#random sampling
train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)

# Data pre-processing

In [7]:
import re

def clean_data(text):
    text= re.sub(r"\r\n"," ",text) #replacing next line space to empty space
    text = re.sub(r"\s+"," ",text) #spaces
    text = re.sub(r"<.*?>"," ",text) #html tags <bvofisbbv>
    text= text.strip().lower()

    return text

In [8]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [9]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

# Tokenization

In [10]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [11]:
# raw data --> tokens
def tokenize(data):
    inputs = tokenizer(data["dialogue"],padding="max_length", max_length=512 , truncation = True)
    targets = tokenizer(data["summary"], padding = "max_length", max_length = 150 , truncation = True)

    inputs["labels"] = targets["input_ids"] # token ids ==> add to input as labels
    return inputs



In [12]:
train_dataset =train_data.apply(tokenize,axis=1).tolist()
val_dataset = val_data.apply(tokenize,axis=1).tolist()


In [13]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [14]:
# input ids - dialogue ===> token ids

#1 ===> End of sequence & 0===> padding 

# attention mask (k)
# labels -- target ==> summary token

In [15]:
type(train_dataset)

list

# Working With the Model

In [16]:
#NLP  ===> generation task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [17]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device" ,device)
model.to(device)


device cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [19]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs = 6,
    weight_decay=0.01,
    per_device_train_batch_size =8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
)

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset = val_dataset
    
)

In [20]:
#train the model
trainer.train()

# saveing the model

In [21]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [18]:
#using saved model

model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

# Testing the core logic for summarization

In [19]:
def summarize_dialogue(dialogue):
    dialogue= clean_data(dialogue) # clean

    #tokenize
    inputs = tokenizer(
        dialogue,
        padding = "max_length",
        max_length= 512,
        truncation=True,
        return_tensors="pt"
    )
    
    #generating the summamry => token ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length=150,
        num_beams=4,  # best of 4 outputs 
        early_stopping=True
    ) 
    
    #token ids convert to summary ==> decoding
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) #EOS , SEP

    return summary

In [20]:
test_dialogue= """
John: Hey Sarah, did you finish the project report?

Sarah: Not yet. I completed the research section, but I still need to work on the results and conclusion.

John: I finished my part yesterday. Do you want me to help you with the results section?

Sarah: That would be great. If we finish it today, we can review the complete report tomorrow.

John: Sure. Let's work on it together this afternoon and submit the report before the deadline.
"""

summary = summarize_dialogue(test_dialogue)

print("summary :", summary)

summary : sarah finished the project report yesterday, but he still needs to work on the results and conclusion.
